# Mimamori AI

Routine deviation detection, evidence grounded explanation, and sentence level verification on CASAS smart home sensor data.


## 1. Setup


In [1]:
import os
import re
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

RESIDENT_ID = "R001"
BASELINE_WINDOW_DAYS = 28
MIN_BASELINE_DAYS = 14
MAD_FLOOR = 1.0
STRONG_Z = 4.0
MODERATE_Z = 3.0
EARLIEST_RISE_HOUR = 4
NIGHT_END_HOUR = 6
VISIT_GAP_MINUTES = 10
SEQUENCE_LENGTH = 7
TRAIN_FRACTION = 0.6
TARGET_FALSE_ALARM_RATE = 0.05
EVALUATION_SAMPLE = 30
HIGH_QUANTILE = 0.97
MEDIUM_QUANTILE = 0.93
REMOTE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
RANDOM_SEED = 7

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
print("pandas", pd.__version__)
print("output directory", OUTPUT_DIR.resolve())


pandas 2.3.3
output directory /kaggle/working


## 2. Inspect what is attached


In [2]:
def preview(path, lines=3, width=112):
    out = []
    try:
        with open(path, "r", errors="ignore") as handle:
            for line in handle:
                if line.strip():
                    out.append(line.strip()[:width])
                if len(out) >= lines:
                    break
    except OSError:
        pass
    return out


def inspect_inputs(root="/kaggle/input", limit=30):
    base = Path(root)
    if not base.is_dir():
        print(root, "does not exist here")
        return
    files = sorted((p for p in base.rglob("*") if p.is_file()), key=lambda p: -p.stat().st_size)
    print(len(files), "files under", root)
    print()
    for path in files[:limit]:
        print(f"{path.stat().st_size / 1048576:8.2f} MB   {path}")
        for line in preview(path):
            print("             |", line)
        print()


inspect_inputs()


1 files under /kaggle/input

   58.29 MB   /kaggle/input/datasets/mdsajjadullah/casas-aruba-activity-labeled/aruba.txt
             | 2010-11-04 00:03:50.209589 M003 ON Sleeping begin
             | 2010-11-04 00:03:57.399391 M003 OFF
             | 2010-11-04 00:15:08.984841 T002 21.5



## 3. Detect the file layout and load the events


In [3]:
DATE_TOKEN = re.compile(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}$")
TIME_TOKEN = re.compile(r"^\d{1,2}:\d{2}(:\d{2})?(\.\d+)?$")
STAMP_TOKEN = re.compile(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}[ T]\d{1,2}:\d{2}")
SENSOR_TOKEN = re.compile(r"^[A-Za-z]{0,12}[_\-]?\d{1,5}$")
ACTIVITY_TOKEN = re.compile(r"^[A-Za-z][A-Za-z_]{2,40}$")
BINARY_VALUES = {"on", "off", "open", "close", "closed", "present", "absent", "true", "false"}


def is_number(text):
    try:
        float(text)
        return True
    except ValueError:
        return False


def split_line(line, delimiter):
    parts = line.split() if delimiter is None else line.split(delimiter)
    return [part.strip() for part in parts]


def read_probe(path, count=300):
    rows = []
    try:
        with open(path, "r", errors="ignore") as handle:
            for line in handle:
                if line.strip():
                    rows.append(line.rstrip("\n"))
                if len(rows) >= count:
                    break
    except OSError:
        return []
    return rows


def pick_delimiter(sample):
    best_score, best = 0.0, None
    for delimiter in (None, ",", "\t", ";", "|"):
        widths = [len(split_line(line, delimiter)) for line in sample]
        common = max(set(widths), key=widths.count)
        if common < 3:
            continue
        score = widths.count(common) / len(widths) * min(common, 8)
        if score > best_score:
            best_score, best = score, delimiter
    return best


def stamp_columns(rows):
    if not rows:
        return None
    if len(rows[0]) >= 2:
        paired = sum(1 for row in rows if DATE_TOKEN.match(row[0]) and TIME_TOKEN.match(row[1]))
        if paired / len(rows) > 0.9:
            return (0, 1)
    single = sum(1 for row in rows if STAMP_TOKEN.match(row[0]))
    if single / len(rows) > 0.9:
        return (0,)
    return None


def detect_layout(path):
    sample = read_probe(path)
    if len(sample) < 30:
        return None
    delimiter = pick_delimiter(sample)
    rows = [split_line(line, delimiter) for line in sample]
    widths = [len(row) for row in rows]
    width = max(value for value in set(widths) if widths.count(value) >= 2)
    rows = [(row + [""] * width)[:width] for row in rows if len(row) >= 3]
    header = 0
    stamp = stamp_columns(rows)
    if stamp is None and len(rows) > 1:
        rows = rows[1:]
        header = 1
        stamp = stamp_columns(rows)
    if stamp is None:
        return None
    tail = [index for index in range(width) if index >= len(stamp)]
    sensor = None
    for index in tail:
        values = [row[index] for row in rows if row[index]]
        if not values:
            continue
        share = sum(1 for value in values if SENSOR_TOKEN.match(value)) / len(values)
        if share > 0.8 and 1 < len(set(values)) <= 500:
            sensor = index
            break
    if sensor is None:
        return None
    reading = None
    for index in tail:
        if index == sensor:
            continue
        values = [row[index].lower() for row in rows if row[index]]
        if not values:
            continue
        share = sum(1 for value in values if value in BINARY_VALUES or is_number(value)) / len(values)
        if share > 0.8:
            reading = index
            break
    if reading is None:
        return None
    extras = [index for index in tail if index not in (sensor, reading)]
    activity = marker = None
    for index in extras:
        values = [row[index] for row in rows if row[index]]
        if not values:
            continue
        if sum(1 for value in values if value.lower() in {"begin", "end"}) / len(values) > 0.5:
            marker = index
        elif sum(1 for value in values if ACTIVITY_TOKEN.match(value)) / len(values) > 0.5:
            activity = index
    return {"delimiter": delimiter, "width": width, "header": header, "stamp": stamp,
            "sensor": sensor, "reading": reading, "activity": activity, "marker": marker}


def discover_dataset(minimum_bytes=100_000):
    roots = [p for p in ["/kaggle/input", os.environ.get("CASAS_DIR", ""), "."] if p and Path(p).is_dir()]
    found = {}
    for root in roots:
        for path in Path(root).rglob("*"):
            if not path.is_file():
                continue
            try:
                size = path.stat().st_size
            except OSError:
                continue
            if size < minimum_bytes:
                continue
            layout = detect_layout(path)
            if layout is None:
                continue
            bonus = 10 ** 12 if "aruba" in str(path).lower() else 0
            bonus += 10 ** 11 if layout["activity"] is not None else 0
            found[str(path.resolve())] = (size + bonus, layout, size)
    if not found:
        raise FileNotFoundError(
            "No sensor event file was recognised. The pipeline needs a file whose rows carry a timestamp, "
            "a sensor identifier and a reading, for example: 2010-11-04 00:03:50.209589  M003  ON. "
            "Attach the raw CASAS Aruba event file through Add Input, or set CASAS_DIR."
        )
    ranked = sorted(found.items(), key=lambda item: -item[1][0])
    for path, (_, layout, size) in ranked[:6]:
        tags = []
        tags.append("delimiter=" + ("whitespace" if layout["delimiter"] is None else repr(layout["delimiter"])))
        tags.append("labelled" if layout["activity"] is not None else "no labels")
        print(f"{size / 1048576:7.2f} MB  {'  '.join(tags):<34} {path}")
    return Path(ranked[0][0]), ranked[0][1][1]


DATA_PATH, LAYOUT = discover_dataset()
print()
print("selected", DATA_PATH)
print("layout  ", LAYOUT)


  58.29 MB  delimiter=whitespace  labelled     /kaggle/input/datasets/mdsajjadullah/casas-aruba-activity-labeled/aruba.txt

selected /kaggle/input/datasets/mdsajjadullah/casas-aruba-activity-labeled/aruba.txt
layout   {'delimiter': None, 'width': 6, 'header': 0, 'stamp': (0, 1), 'sensor': 2, 'reading': 3, 'activity': 4, 'marker': 5}


In [4]:
def load_events(path, layout):
    delimiter = layout["delimiter"]
    width = layout["width"]
    stamp = layout["stamp"]
    keep = []
    with open(path, "r", errors="ignore") as handle:
        for number, line in enumerate(handle):
            if number < layout["header"] or not line.strip():
                continue
            parts = split_line(line.rstrip("\n"), delimiter)
            if len(parts) < 3:
                continue
            keep.append((parts + [""] * width)[:width])
    if not keep:
        raise ValueError(f"{path} produced no usable rows with the detected layout")
    frame = pd.DataFrame(keep, columns=[f"c{index}" for index in range(width)])
    if len(stamp) == 2:
        raw = frame[f"c{stamp[0]}"] + " " + frame[f"c{stamp[1]}"]
    else:
        raw = frame[f"c{stamp[0]}"].str.replace("T", " ", regex=False)
    frame["ts"] = pd.to_datetime(raw.str.slice(0, 19), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    if frame["ts"].isna().mean() > 0.5:
        frame["ts"] = pd.to_datetime(raw, errors="coerce")
    frame["sensor"] = frame[f"c{layout['sensor']}"].astype(str).str.strip().str.upper()
    frame["value"] = frame[f"c{layout['reading']}"].astype(str).str.strip()
    frame["activity"] = frame[f"c{layout['activity']}"] if layout["activity"] is not None else None
    frame["marker"] = frame[f"c{layout['marker']}"] if layout["marker"] is not None else None
    frame = frame[["ts", "sensor", "value", "activity", "marker"]].dropna(subset=["ts"])
    frame["kind"] = frame["sensor"].str[0]
    return frame.sort_values("ts").reset_index(drop=True)


def carry_activity(frame):
    current = None
    filled = []
    for activity, marker in zip(frame["activity"].tolist(), frame["marker"].tolist()):
        tag = marker.lower() if isinstance(marker, str) else ""
        clean = activity.strip() if isinstance(activity, str) and activity.strip() else None
        if clean and tag.startswith("b"):
            current = clean
            filled.append(clean)
        elif clean and tag.startswith("e"):
            filled.append(clean)
            current = None
        elif clean and not tag:
            filled.append(clean)
        else:
            filled.append(current)
    return filled


def choose_motion(frame):
    binary = frame[frame["value"].str.lower().isin(BINARY_VALUES)]
    if not binary.empty and binary["sensor"].str.startswith("M").any():
        return binary[binary["sensor"].str.startswith("M")].copy()
    if not binary.empty:
        counts = binary.groupby("sensor").size()
        busiest = counts[counts >= counts.quantile(0.2)].index
        return binary[binary["sensor"].isin(busiest)].copy()
    return frame[frame["kind"] == "M"].copy()


events = load_events(DATA_PATH, LAYOUT)
events["labelled_activity"] = carry_activity(events)
events["day"] = events["ts"].dt.normalize()
motion = choose_motion(events)

print("events        ", f"{len(events):,}")
print("motion events ", f"{len(motion):,}")
print("distinct days ", events["day"].nunique())
print("range         ", events["ts"].min(), "to", events["ts"].max())
print("sensors       ", events["sensor"].nunique())
print("labelled rows ", int(events["labelled_activity"].notna().sum()))

if motion.empty:
    raise ValueError(
        "No motion style events were found. This file does not carry the room level movement data the "
        "pipeline needs. Attach the raw CASAS Aruba event file instead."
    )
if events["day"].nunique() < 40:
    print()
    print("warning: fewer than 40 days are present, so a 28 day rolling baseline will cover very little of the record")


events         1,719,558
motion events  1,595,987
distinct days  220
range          2010-11-04 00:03:50 to 2011-06-11 23:58:10
sensors        41
labelled rows  793153


## 4. Derive the sensor to room map from the labels


In [5]:
ACTIVITY_TO_ZONE = {
    "Sleeping": "Bedroom",
    "Bed_to_Toilet": "Bathroom",
    "Bed_To_Toilet": "Bathroom",
    "Meal_Preparation": "Kitchen",
    "Wash_Dishes": "Kitchen",
    "Eating": "Dining",
    "Relax": "Living Room",
    "Work": "Office",
    "Enter_Home": "Entrance",
    "Leave_Home": "Entrance",
    "Housekeeping": "Living Room",
    "Respirate": "Other",
    "Resperate": "Other",
}
ZONES = ["Bedroom", "Bathroom", "Kitchen", "Dining", "Living Room", "Office", "Entrance", "Other"]


def build_zone_map(frame):
    hint = frame["labelled_activity"].map(ACTIVITY_TO_ZONE)
    mask = hint.notna()
    mapping = {}
    if mask.any():
        labelled = pd.DataFrame({"sensor": frame.loc[mask, "sensor"].values, "zone": hint[mask].values})
        mapping = pd.crosstab(labelled["sensor"], labelled["zone"]).idxmax(axis=1).to_dict()
    for sensor in frame["sensor"].unique():
        mapping.setdefault(sensor, "Other")
    return mapping


ZONE_MAP = build_zone_map(motion)
motion["zone"] = motion["sensor"].map(ZONE_MAP)

for zone in ZONES:
    members = sorted(name for name, value in ZONE_MAP.items() if value == zone)
    if members:
        print(f"{zone:<12} {', '.join(members)}")

named = sum(1 for value in ZONE_MAP.values() if value != "Other")
print()
print(f"sensors mapped to a named room: {named} of {len(ZONE_MAP)}")
if named < 3:
    print("warning: this file carries few or no activity labels, so room names cannot be derived from it.")
    print("the pipeline will still run, but the alerts will refer to movement in general rather than to rooms.")
    print("attach the labelled CASAS Aruba event file to get room level explanations.")


Bedroom      M002, M003, M007
Bathroom     M004
Kitchen      M014, M015, M016, M017, M018, M019, M021, M022, M023
Living Room  M001, M005, M006, M008, M009, M010, M011, M012, M013, M020, M024, M028, M031
Office       M025, M026, M027
Entrance     M029, M030

sensors mapped to a named room: 31 of 31


## 5. Build one feature row per day


In [6]:
motion["minute"] = motion["ts"].dt.floor("min")
occupancy = motion.drop_duplicates(subset=["minute", "zone"])[["ts", "minute", "day", "zone"]].copy()

zone_minutes = (
    occupancy.groupby(["day", "zone"]).size().unstack(fill_value=0).reindex(columns=ZONES, fill_value=0)
)
zone_minutes.columns = [f"minutes_{name.lower().replace(' ', '_')}" for name in zone_minutes.columns]


def minutes_from_midnight(series):
    return (series - series.dt.normalize()).dt.total_seconds() / 60.0


def count_visits(times):
    if len(times) == 0:
        return 0
    ordered = times.sort_values()
    gaps = ordered.diff().dt.total_seconds().div(60).fillna(10 ** 9)
    return int((gaps > VISIT_GAP_MINUTES).sum())


out_of_bed = ~occupancy["zone"].isin(["Bedroom", "Bathroom"])
away_from_bed = occupancy[out_of_bed & (occupancy["ts"].dt.hour >= EARLIEST_RISE_HOUR)]
first_exit = minutes_from_midnight(away_from_bed.groupby("day")["minute"].min()).rename("first_exit_minutes")

night_bathroom = occupancy[(occupancy["zone"] == "Bathroom") & (occupancy["ts"].dt.hour < NIGHT_END_HOUR)]
night_visits = night_bathroom.groupby("day")["minute"].apply(count_visits).rename("night_bathroom_visits")

daily_events = motion.groupby("day").size().rename("motion_events")

features = zone_minutes.join(first_exit, how="outer").join(night_visits, how="outer").join(daily_events, how="outer")
features["first_exit_minutes"] = features["first_exit_minutes"].ffill().bfill()
features = features.fillna(0).sort_index()
if len(features) > 6:
    features = features.iloc[1:-1]
spread_by_column = features.std(numeric_only=True)
useful = [name for name in features.columns if float(spread_by_column.get(name, 0) or 0) > 0]
if not useful:
    raise ValueError("No daily feature varied across the record. Check that the attached file is a raw event log.")
features = features[useful]

FEATURE_COLUMNS = list(features.columns)
print("days", len(features))
print("features", len(FEATURE_COLUMNS))
features.tail(5)


days 218
features 9


,minutes_bedroom,minutes_bathroom,minutes_kitchen,minutes_living_room,minutes_office,minutes_entrance,first_exit_minutes,night_bathroom_visits,motion_events
day,,,,,,,,,
2011-06-06,155,52,253,527,37,30,335.0,2.0,9246
2011-06-07,78,35,216,377,0,21,329.0,2.0,6780
2011-06-08,109,46,260,459,17,17,307.0,3.0,7638
2011-06-09,86,47,142,309,1,16,309.0,3.0,5094
2011-06-10,99,33,216,338,34,33,324.0,1.0,7520


## 6. Rolling baseline and the statistical detector


In [7]:
def rolling_median(frame):
    return frame.rolling(BASELINE_WINDOW_DAYS, min_periods=MIN_BASELINE_DAYS).median().shift(1)


def rolling_mad(frame):
    def deviation(values):
        centre = np.median(values)
        return np.median(np.abs(values - centre))

    return frame.rolling(BASELINE_WINDOW_DAYS, min_periods=MIN_BASELINE_DAYS).apply(deviation, raw=True).shift(1)


BASELINE_MEDIAN = rolling_median(features)
BASELINE_MAD = rolling_mad(features)


def robust_z(frame, median, mad):
    return 0.6745 * (frame - median) / mad.clip(lower=MAD_FLOOR)


DEVIATION = robust_z(features, BASELINE_MEDIAN, BASELINE_MAD)
scored = DEVIATION.dropna(how="all")


CONCERNING_DIRECTION = {
    "minutes_bedroom": 1,
    "minutes_bathroom": 1,
    "minutes_kitchen": -1,
    "minutes_dining": -1,
    "minutes_living_room": -1,
    "first_exit_minutes": 1,
    "night_bathroom_visits": 1,
    "motion_events": -1,
}
DIRECTION_VECTOR = np.array([CONCERNING_DIRECTION.get(name, 0) for name in FEATURE_COLUMNS], dtype=float)
WEIGHTS = pd.Series(CONCERNING_DIRECTION).reindex(FEATURE_COLUMNS).fillna(0)
ALERTING_COLUMNS = [name for name in FEATURE_COLUMNS if WEIGHTS[name] != 0]


def combined_statistic(concern_row):
    positive = np.clip(np.asarray(concern_row, dtype=float), 0, None)
    return float(np.sqrt(np.nansum(positive ** 2)))


CONCERN = (scored[ALERTING_COLUMNS] * WEIGHTS[ALERTING_COLUMNS]).dropna()
COMBINED = CONCERN.apply(combined_statistic, axis=1)
HIGH_CUT = float(COMBINED.quantile(HIGH_QUANTILE))
MEDIUM_CUT = float(COMBINED.quantile(MEDIUM_QUANTILE))


def classify(day):
    row = CONCERN.loc[day]
    value = float(COMBINED.loc[day])
    if value >= HIGH_CUT:
        level = "High"
    elif value >= MEDIUM_CUT:
        level = "Medium"
    else:
        level = "Normal"
    return pd.Series({
        "severity": level,
        "combined": round(value, 2),
        "peak_z": round(float(row.max()), 2),
        "driver": row.idxmax(),
        "moderate_count": int((row >= MODERATE_Z).sum()),
    })


verdicts = pd.DataFrame([classify(day) for day in CONCERN.index], index=CONCERN.index)
alerts = verdicts[verdicts["severity"] != "Normal"].sort_values("combined", ascending=False)

print(f"thresholds calibrated from this resident: medium {MEDIUM_CUT:.2f}   high {HIGH_CUT:.2f}")
print()

if alerts.empty:
    alerts = verdicts.dropna(subset=["peak_z"]).sort_values("peak_z", ascending=False).head(5).copy()
    alerts["severity"] = "Medium"
    print("no day crossed the thresholds, using the five most deviant days so the rest of the notebook still runs")
    print()

print(verdicts["severity"].value_counts().to_string())
print()
print("alert rate", f"{len(alerts) / max(1, len(verdicts)):.1%}")
print()
print(alerts.head(10).to_string())


thresholds calibrated from this resident: medium 3.72   high 4.98

severity
Normal    189
Medium      8
High        7

alert rate 7.4%

           severity  combined  peak_z              driver  moderate_count
day                                                                      
2010-12-06     High      6.67    5.44     minutes_bedroom               2
2011-05-29     High      5.77    4.72    minutes_bathroom               2
2010-12-01     High      5.56    5.47  first_exit_minutes               1
2011-06-06     High      5.16    3.76     minutes_bedroom               2
2011-05-01     High      5.10    4.66     minutes_bedroom               1
2011-02-14     High      5.07    3.99    minutes_bathroom               2
2011-03-20     High      5.03    4.69     minutes_bedroom               1
2011-03-12   Medium      4.41    3.85    minutes_bathroom               1
2010-12-21   Medium      4.39    4.39  first_exit_minutes               1
2011-04-17   Medium      4.23    2.98     minutes_

## 7. GRU autoencoder as a comparison detector


In [8]:
import torch
from torch import nn

torch.manual_seed(RANDOM_SEED)

matrix = features[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
split_index = max(SEQUENCE_LENGTH + 5, int(len(matrix) * TRAIN_FRACTION))
centre = matrix[:split_index].mean(axis=0)
spread = matrix[:split_index].std(axis=0)
spread[spread < 1e-6] = 1.0
scaled = (matrix - centre) / spread


def make_windows(source, length):
    return np.stack([source[i - length + 1: i + 1] for i in range(length - 1, len(source))])


windows = make_windows(scaled, SEQUENCE_LENGTH)
window_days = features.index[SEQUENCE_LENGTH - 1:]
train_windows = windows[: split_index - SEQUENCE_LENGTH + 1]


class SequenceAutoencoder(nn.Module):
    def __init__(self, n_features, hidden=32):
        super().__init__()
        self.encoder = nn.GRU(n_features, hidden, batch_first=True)
        self.decoder = nn.GRU(hidden, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_features)

    def forward(self, batch):
        _, state = self.encoder(batch)
        seeded = state[-1].unsqueeze(1).repeat(1, batch.size(1), 1)
        decoded, _ = self.decoder(seeded)
        return self.head(decoded)


model = SequenceAutoencoder(len(FEATURE_COLUMNS))
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
train_tensor = torch.tensor(train_windows)

model.train()
for epoch in range(600):
    optimiser.zero_grad()
    loss = criterion(model(train_tensor), train_tensor)
    loss.backward()
    optimiser.step()
    if (epoch + 1) % 150 == 0:
        print(f"epoch {epoch + 1:3d}   loss {loss.item():.4f}")

model.eval()


def reconstruction_error(batch):
    with torch.no_grad():
        tensor = torch.tensor(np.asarray(batch, dtype=np.float32))
        rebuilt = model(tensor)
        return ((rebuilt[:, -1, :] - tensor[:, -1, :]) ** 2).mean(dim=1).numpy()


gru_scores = pd.Series(reconstruction_error(windows), index=window_days, name="gru_score")
print()
print(gru_scores.describe().to_string())


epoch 150   loss 0.7242
epoch 300   loss 0.5396
epoch 450   loss 0.3483
epoch 600   loss 0.2384

count    212.000000
mean       0.720526
std        1.100054
min        0.027616
25%        0.103412
50%        0.263728
75%        0.978928
max        9.733516


## 8. Evaluation with injected anomalies


In [9]:
PERTURBATIONS = ["skipped_meals", "late_rise", "low_mobility", "restless_night"]


def perturb(row, columns, kind):
    changed = row.copy()
    position = {name: i for i, name in enumerate(columns)}
    if kind == "skipped_meals":
        for name in ("minutes_kitchen", "minutes_dining"):
            if name in position:
                changed[position[name]] *= 0.1
    elif kind == "late_rise":
        if "first_exit_minutes" in position:
            changed[position["first_exit_minutes"]] += 200
        if "minutes_bedroom" in position:
            changed[position["minutes_bedroom"]] *= 1.6
    elif kind == "low_mobility":
        for name in columns:
            if name.startswith("minutes_") and name != "minutes_bedroom":
                changed[position[name]] *= 0.3
        if "motion_events" in position:
            changed[position["motion_events"]] *= 0.35
    elif kind == "restless_night":
        if "night_bathroom_visits" in position:
            changed[position["night_bathroom_visits"]] += 5
        if "minutes_bathroom" in position:
            changed[position["minutes_bathroom"]] *= 2.5
    return changed


usable_baseline = set(BASELINE_MEDIAN.dropna(how="any").index)
position_of_day = {day: index for index, day in enumerate(features.index)}
window_position = {day: index for index, day in enumerate(window_days)}
evaluation_days = [
    day for day in window_days
    if day in usable_baseline and position_of_day[day] >= split_index
]


def statistical_score(row, day):
    median = BASELINE_MEDIAN.loc[day, FEATURE_COLUMNS].to_numpy(dtype=float)
    mad = np.clip(BASELINE_MAD.loc[day, FEATURE_COLUMNS].to_numpy(dtype=float), MAD_FLOOR, None)
    concern = 0.6745 * (row - median) / mad * DIRECTION_VECTOR
    concern[DIRECTION_VECTOR == 0] = 0.0
    return combined_statistic(concern)


def gru_score_for(row, day):
    window = windows[window_position[day]].copy()
    window[-1] = (row - centre) / spread
    return float(reconstruction_error(window[None, ...])[0])


clean_statistical = np.array([statistical_score(features.loc[day, FEATURE_COLUMNS].to_numpy(dtype=float), day) for day in evaluation_days])
clean_gru = np.array([gru_scores.loc[day] for day in evaluation_days])

statistical_cut = float(np.quantile(clean_statistical, 1 - TARGET_FALSE_ALARM_RATE))
gru_cut = float(np.quantile(clean_gru, 1 - TARGET_FALSE_ALARM_RATE))

from scipy.stats import rankdata


def area_under_curve(negative, positive):
    scores = np.concatenate([np.asarray(negative), np.asarray(positive)])
    ranks = rankdata(scores)
    positive_ranks = ranks[len(negative):].sum()
    count_positive = len(positive)
    count_negative = len(negative)
    return float((positive_ranks - count_positive * (count_positive + 1) / 2) / (count_positive * count_negative))


rows = []
for kind in PERTURBATIONS:
    altered_statistical = []
    altered_gru = []
    for day in evaluation_days:
        original = features.loc[day, FEATURE_COLUMNS].to_numpy(dtype=float)
        altered = perturb(original, FEATURE_COLUMNS, kind)
        altered_statistical.append(statistical_score(altered, day))
        altered_gru.append(gru_score_for(altered, day))
    altered_statistical = np.array(altered_statistical)
    altered_gru = np.array(altered_gru)
    rows.append({
        "injected_anomaly": kind,
        "stat_auc": round(area_under_curve(clean_statistical, altered_statistical), 3),
        "stat_recall": round(float((altered_statistical > statistical_cut).mean()), 3),
        "gru_auc": round(area_under_curve(clean_gru, altered_gru), 3),
        "gru_recall": round(float((altered_gru > gru_cut).mean()), 3),
        "days": len(evaluation_days),
    })

comparison = pd.DataFrame(rows)
comparison.loc[len(comparison)] = {
    "injected_anomaly": "mean",
    "stat_auc": round(comparison["stat_auc"].mean(), 3),
    "stat_recall": round(comparison["stat_recall"].mean(), 3),
    "gru_auc": round(comparison["gru_auc"].mean(), 3),
    "gru_recall": round(comparison["gru_recall"].mean(), 3),
    "days": len(evaluation_days),
}

print(f"recall measured with the false alarm rate held at {TARGET_FALSE_ALARM_RATE:.0%} on unperturbed days")
print(f"auc needs no threshold, 0.5 is chance and 1.0 is perfect")
print(f"statistical cut {statistical_cut:.2f}   gru cut {gru_cut:.4f}")
print()
print(comparison.to_string(index=False))


recall measured with the false alarm rate held at 5% on unperturbed days
auc needs no threshold, 0.5 is chance and 1.0 is perfect
statistical cut 4.18   gru cut 3.4271

injected_anomaly  stat_auc  stat_recall  gru_auc  gru_recall  days
   skipped_meals     0.883        0.125    0.651       0.091    88
       late_rise     0.931        0.511    0.666       0.227    88
    low_mobility     0.949        0.602    0.830       0.227    88
  restless_night     0.995        0.966    0.967       0.875    88
            mean     0.940        0.551    0.779       0.355    88


## 9. Evidence records


In [10]:
ZONE_FOR_COLUMN = {f"minutes_{name.lower().replace(' ', '_')}": name for name in ZONES}
ROOM_FOR_DRIVER = {
    "first_exit_minutes": "Bedroom",
    "night_bathroom_visits": "Bathroom",
    "motion_events": "the whole home",
}
PLAIN_NAME = {
    "first_exit_minutes": "time of first movement outside the bedroom",
    "night_bathroom_visits": "bathroom visits during the night",
    "motion_events": "overall movement recorded in the home",
}


def as_clock(minutes):
    if minutes is None or not np.isfinite(minutes):
        return None
    total = int(round(minutes)) % (24 * 60)
    return f"{total // 60:02d}:{total % 60:02d}"


def describe(column, value):
    if column == "first_exit_minutes":
        return as_clock(value)
    if column == "night_bathroom_visits":
        return f"{int(round(value))} visits"
    if column == "motion_events":
        return f"{int(round(value))} sensor events"
    return f"{int(round(value))} minutes"


def quiet_zones(day):
    today = features.loc[day]
    usual = BASELINE_MEDIAN.loc[day]
    missing = []
    for column, zone in ZONE_FOR_COLUMN.items():
        if column not in features.columns or zone == "Other":
            continue
        typical = float(usual.get(column, 0) or 0)
        if typical >= 20 and float(today[column]) <= max(2.0, 0.2 * typical):
            missing.append(zone)
    return missing


def build_evidence(day):
    verdict = verdicts.loc[day]
    driver = verdict["driver"]
    observed = float(features.loc[day, driver])
    expected = float(BASELINE_MEDIAN.loc[day, driver])
    return {
        "date": str(pd.Timestamp(day).date()),
        "resident": RESIDENT_ID,
        "room": ZONE_FOR_COLUMN.get(driver) or ROOM_FOR_DRIVER.get(driver, "the whole home"),
        "measure": PLAIN_NAME.get(driver, f"time spent in the {ZONE_FOR_COLUMN.get(driver, 'home')}"),
        "observed": describe(driver, observed),
        "usual": describe(driver, expected),
        "deviation_sigma": round(float(verdict["peak_z"]), 2),
        "quiet_rooms": quiet_zones(day),
        "baseline_window_days": BASELINE_WINDOW_DAYS,
        "severity": verdict["severity"],
        "known_context": None,
    }


EVIDENCE_RECORDS = {str(pd.Timestamp(day).date()): build_evidence(day) for day in alerts.index}
sample_day = alerts.index[0]
print(json.dumps(build_evidence(sample_day), indent=2))


{
  "date": "2010-12-06",
  "resident": "R001",
  "room": "Bedroom",
  "measure": "time spent in the Bedroom",
  "observed": "166 minutes",
  "usual": "102 minutes",
  "deviation_sigma": 5.44,
  "quiet_rooms": [],
  "baseline_window_days": 28,
  "severity": "High",
  "known_context": null
}


## 10. Message generation


In [11]:
SYSTEM_PROMPT = (
    "You write short alerts for the care team of an elderly person living alone. "
    "Use only the facts in the evidence given to you. "
    "Never state or suggest a cause such as a fall, an illness or an injury. "
    "Never invent numbers or times. "
    "Write plainly for a family member. Do not mention statistics, sigma, standard deviations, "
    "confidence or percentages. "
    "Write at most three short sentences. "
    "End with exactly this sentence: Please check on the resident."
)
PERMISSIVE_PROMPT = (
    "You are a helpful assistant for an elderly care team. "
    "Look at the sensor evidence below and explain to the caregiver what happened, "
    "what it probably means, and what they should do. Be warm and reassuring."
)
PROMPT_HIDDEN_FIELDS = {"deviation_sigma", "resident", "known_context"}
PROMPT_CONDITIONS = {"production": SYSTEM_PROMPT, "permissive": PERMISSIVE_PROMPT}


def build_user_prompt(evidence):
    lines = [
        f"{key}: {value}" for key, value in evidence.items()
        if key not in PROMPT_HIDDEN_FIELDS and value not in (None, [], "")
    ]
    return "Evidence\n" + "\n".join(lines) + "\n\nWrite the alert."


def read_token():
    for name in ("HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = os.environ.get(name)
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def remote_message(evidence, system_prompt, attempts=3):
    from huggingface_hub import InferenceClient

    client = InferenceClient(model=REMOTE_MODEL, token=read_token(), timeout=60)
    failure = None
    for attempt in range(attempts):
        try:
            reply = client.chat_completion(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": build_user_prompt(evidence)},
                ],
                max_tokens=180,
                temperature=0.3,
            )
            return reply.choices[0].message.content.strip()
        except Exception as error:
            failure = error
            time.sleep(3 * (attempt + 1))
    raise failure


def local_message(evidence, loose=False):
    parts = [
        f"The {evidence['measure']} was {evidence['observed']} on {evidence['date']}.",
        f"Over the last {evidence['baseline_window_days']} days the usual value is {evidence['usual']}.",
    ]
    if evidence["quiet_rooms"]:
        parts.append("There was almost no activity in the " + " and the ".join(r.lower() for r in evidence["quiet_rooms"]) + ".")
    if loose:
        parts.append("The resident may have had a fall during the night.")
        parts.append("This pattern often points to an illness starting.")
        parts.append("Movement was around 45 percent below the usual level.")
    parts.append("Please check on the resident.")
    return " ".join(parts)


def generate_message(evidence, system_prompt, loose=False):
    try:
        return remote_message(evidence, system_prompt), "remote"
    except Exception as error:
        return local_message(evidence, loose), f"local ({type(error).__name__})"


sample_evidence = EVIDENCE_RECORDS[str(pd.Timestamp(sample_day).date())]
drafts = {}
for label, prompt in PROMPT_CONDITIONS.items():
    text, origin = generate_message(sample_evidence, prompt, loose=(label == "permissive"))
    drafts[label] = text
    print(f"[{label}]  source: {origin}")
    print(text)
    print()


[production]  source: remote
The resident spent 166 minutes in the Bedroom, which is more than usual. Please check on the resident.

[permissive]  source: remote
Dear Caregiver,

I wanted to reach out to you with some information about Mrs. Johnson. Based on the sensor data, it appears that she has spent significantly more time in her bedroom today compared to her usual routine. Specifically, she has been in the bedroom for 166 minutes, which is much higher than her usual 102 minutes.

This increase in time spent in the bedroom could indicate a few different things. She might be feeling unwell, needing rest, or perhaps she's just engaging in some activities that she enjoys. However, given that the severity level is marked as "High," it's important that we check in on her to ensure she's doing well.

Please make sure to stop by her room or give her a call to see if everything is alright. If she seems unwell or if you have any concerns, please let me know so we can provide



## 11. Verification layer


In [12]:
SPECULATIVE_TERMS = [
    "fall", "fell", "fallen", "stroke", "heart", "illness", " ill ", "sick", "injur", "unconscious",
    "emergency", "hospital", "medication", "medicine", "pain", "dementia", "confus", "dizzy",
    "faint", "collapse", "died", "dead", "danger", "depress", "lonely", "suffer", "risk of",
]
APPROVED_CLOSINGS = {
    "please check on the resident.",
    "please contact the resident.",
    "a check is recommended.",
    "this is worth checking.",
}
DATE_PATTERN = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
TIME_PATTERN = re.compile(r"\b\d{1,2}:\d{2}\b")
NUMBER_PATTERN = re.compile(r"\d+(?:\.\d+)?")
SPECULATION_PATTERN = re.compile(
    r"\b(might|may|could|would|should|probably|possibly|perhaps|likely|maybe|seem|seems|"
    r"appear|appears|suggest|suggests|indicate|indicates|presumably|potentially|suspect|"
    r"assume|hopefully|due to|because of|caused by|imply|implies)\b"
)
ZONE_WORDS = [name.lower() for name in ZONES if name != "Other"]


def grounding_terms(evidence):
    text = " ".join(str(value) for value in evidence.values() if value not in (None, [], ""))
    return set(re.findall(r"[a-z]{5,}", text.lower()))


def has_grounding(sentence, evidence):
    lowered = sentence.lower()
    if any(word in lowered for word in grounding_terms(evidence)):
        return True
    return any(str(number) in sentence for number in allowed_numbers(evidence))


def evidence_blob(evidence, drop_date=False):
    payload = {k: v for k, v in evidence.items() if not (drop_date and k == "date")}
    return json.dumps(payload, default=str)


def allowed_dates(evidence):
    return set(DATE_PATTERN.findall(evidence_blob(evidence)))


def allowed_times(evidence):
    return set(TIME_PATTERN.findall(DATE_PATTERN.sub(" ", evidence_blob(evidence))))


def allowed_numbers(evidence):
    text = TIME_PATTERN.sub(" ", DATE_PATTERN.sub(" ", evidence_blob(evidence)))
    return {round(float(value)) for value in NUMBER_PATTERN.findall(text)}


def split_sentences(text):
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text.strip()) if part.strip()]


def check_sentence(sentence, evidence):
    lowered = " " + sentence.lower().strip() + " "
    if sentence.lower().strip() in APPROVED_CLOSINGS:
        return True, "approved closing"
    for term in SPECULATIVE_TERMS:
        if term in lowered:
            return False, f"unsupported cause or judgement: {term.strip()}"
    hedge = SPECULATION_PATTERN.search(lowered)
    if hedge:
        return False, f"speculation beyond the evidence: {hedge.group(1)}"
    permitted_dates = allowed_dates(evidence)
    for value in DATE_PATTERN.findall(sentence):
        if value not in permitted_dates:
            return False, f"date not present in evidence: {value}"
    undated = DATE_PATTERN.sub(" ", sentence)
    permitted_times = allowed_times(evidence)
    for value in TIME_PATTERN.findall(undated):
        if value not in permitted_times:
            return False, f"time not present in evidence: {value}"
    permitted_numbers = allowed_numbers(evidence)
    for value in NUMBER_PATTERN.findall(TIME_PATTERN.sub(" ", undated)):
        if round(float(value)) not in permitted_numbers:
            return False, f"number not present in evidence: {value}"
    blob = evidence_blob(evidence).lower()
    for zone in ZONE_WORDS:
        if zone in lowered and zone not in blob:
            return False, f"room not present in evidence: {zone}"
    if not has_grounding(sentence, evidence):
        return False, "no fact from the evidence appears in this sentence"
    return True, "supported by evidence"


def verify(message, evidence):
    sentences = split_sentences(message)
    kept, dropped = [], []
    for sentence in sentences:
        supported, reason = check_sentence(sentence, evidence)
        (kept if supported else dropped).append({"sentence": sentence, "reason": reason})
    total = len(sentences) or 1
    return {
        "verified_message": " ".join(item["sentence"] for item in kept),
        "removed": dropped,
        "faithfulness": round(len(kept) / total, 3),
        "sentences": len(sentences),
    }


for label, text in drafts.items():
    checked = verify(text, sample_evidence)
    print(f"===== {label} =====")
    print("before  ", text)
    print("after   ", checked["verified_message"])
    print("faithfulness", checked["faithfulness"])
    for item in checked["removed"]:
        print("  removed:", item["sentence"])
        print("           reason:", item["reason"])
    print()


===== production =====
before   The resident spent 166 minutes in the Bedroom, which is more than usual. Please check on the resident.
after    The resident spent 166 minutes in the Bedroom, which is more than usual. Please check on the resident.
faithfulness 1.0

===== permissive =====
before   Dear Caregiver,

I wanted to reach out to you with some information about Mrs. Johnson. Based on the sensor data, it appears that she has spent significantly more time in her bedroom today compared to her usual routine. Specifically, she has been in the bedroom for 166 minutes, which is much higher than her usual 102 minutes.

This increase in time spent in the bedroom could indicate a few different things. She might be feeling unwell, needing rest, or perhaps she's just engaging in some activities that she enjoys. However, given that the severity level is marked as "High," it's important that we check in on her to ensure she's doing well.

Please make sure to stop by her room or give her a cal

## 12. Faithfulness measured across many alerts


In [13]:
sample_days = list(alerts.index[:EVALUATION_SAMPLE])
records = []

for label, prompt in PROMPT_CONDITIONS.items():
    for day in sample_days:
        key = str(pd.Timestamp(day).date())
        evidence = EVIDENCE_RECORDS[key]
        message, origin = generate_message(evidence, prompt, loose=(label == "permissive"))
        outcome = verify(message, evidence)
        records.append({
            "condition": label,
            "date": key,
            "source": origin.split(" ")[0],
            "severity": evidence["severity"],
            "sentences": outcome["sentences"],
            "removed": len(outcome["removed"]),
            "faithfulness": outcome["faithfulness"],
            "raw_message": message,
            "verified_message": outcome["verified_message"],
            "reasons": [item["reason"] for item in outcome["removed"]],
        })
        time.sleep(1)

report = pd.DataFrame(records)

summary_rows = []
for label in PROMPT_CONDITIONS:
    group = report[report["condition"] == label]
    if group.empty:
        continue
    summary_rows.append({
        "condition": label,
        "messages": len(group),
        "sentences": int(group["sentences"].sum()),
        "sentences_removed": int(group["removed"].sum()),
        "messages_flagged": int((group["removed"] > 0).sum()),
        "flagged_share": round(float((group["removed"] > 0).mean()), 3),
        "faithfulness_before": round(float(group["faithfulness"].mean()), 3),
        "faithfulness_after": 1.0,
    })
summary = pd.DataFrame(summary_rows)

print("message source:", report["source"].value_counts().to_dict())
print("alerts per condition:", len(sample_days))
print()
print(summary.to_string(index=False))
print()

rejected = [reason for row in report["reasons"] for reason in row]
if rejected:
    print("what the verifier rejected")
    for reason, count in pd.Series(rejected).value_counts().items():
        print(f"  {count:>3}  {reason}")
else:
    print("nothing was rejected in either condition")

if float((report["source"] == "local").mean()) > 0.3:
    print()
    print("caution: many of these messages came from the local template, which speculates by design.")
    print("the permissive column is only a real measurement when the remote model wrote the text.")


message source: {'remote': 25, 'local': 5}
alerts per condition: 15

 condition  messages  sentences  sentences_removed  messages_flagged  flagged_share  faithfulness_before  faithfulness_after
production        15         32                  0                 0            0.0                1.000                 1.0
permissive        15        120                 83                15            1.0                0.317                 1.0

what the verifier rejected
   29  no fact from the evidence appears in this sentence
   12  speculation beyond the evidence: might
   11  speculation beyond the evidence: could
    4  unsupported cause or judgement: fall
    3  unsupported cause or judgement: illness
    3  number not present in evidence: 45
    2  speculation beyond the evidence: perhaps
    2  speculation beyond the evidence: may
    2  unsupported cause or judgement: emergency
    1  unsupported cause or judgement: confus
    1  speculation beyond the evidence: would
    1  numbe

## 13. Export artifacts for the demo


In [14]:
features.to_csv(OUTPUT_DIR / "daily_features.csv")
BASELINE_MEDIAN.to_csv(OUTPUT_DIR / "baseline_median.csv")
BASELINE_MAD.to_csv(OUTPUT_DIR / "baseline_mad.csv")
DEVIATION.to_csv(OUTPUT_DIR / "deviation_scores.csv")
verdicts.to_csv(OUTPUT_DIR / "daily_verdicts.csv")
comparison.to_csv(OUTPUT_DIR / "detector_comparison.csv", index=False)
report.drop(columns=["reasons"]).to_csv(OUTPUT_DIR / "verification_report.csv", index=False)
summary.to_csv(OUTPUT_DIR / "verification_summary.csv", index=False)

with open(OUTPUT_DIR / "zone_map.json", "w") as handle:
    json.dump(ZONE_MAP, handle, indent=2)

with open(OUTPUT_DIR / "evidence_records.json", "w") as handle:
    json.dump(EVIDENCE_RECORDS, handle, indent=2)

with open(OUTPUT_DIR / "detector_config.json", "w") as handle:
    json.dump({
        "resident": RESIDENT_ID,
        "baseline_window_days": BASELINE_WINDOW_DAYS,
        "min_baseline_days": MIN_BASELINE_DAYS,
        "mad_floor": MAD_FLOOR,
        "strong_z": STRONG_Z,
        "moderate_z": MODERATE_Z,
        "medium_cut": MEDIUM_CUT,
        "high_cut": HIGH_CUT,
        "concerning_direction": CONCERNING_DIRECTION,
        "feature_columns": FEATURE_COLUMNS,
        "zones": ZONES,
        "remote_model": REMOTE_MODEL,
        "statistical_threshold": statistical_cut,
        "gru_threshold": gru_cut,
    }, handle, indent=2)

torch.save({"state_dict": model.state_dict(), "centre": centre, "spread": spread,
            "sequence_length": SEQUENCE_LENGTH, "features": FEATURE_COLUMNS},
           OUTPUT_DIR / "gru_autoencoder.pt")

WRITTEN = [
    "daily_features.csv", "baseline_median.csv", "baseline_mad.csv", "deviation_scores.csv",
    "daily_verdicts.csv", "detector_comparison.csv", "verification_report.csv",
    "verification_summary.csv", "zone_map.json", "evidence_records.json",
    "detector_config.json", "gru_autoencoder.pt",
]

print("written to", OUTPUT_DIR.resolve())
for name in WRITTEN:
    path = OUTPUT_DIR / name
    if path.exists():
        print(f"  {name:<28} {path.stat().st_size / 1024:8.1f} KB")

print()
print("summary")
print(f"  days analysed              {len(features)}")
print(f"  alerts raised              {len(alerts)}  ({len(alerts) / max(1, len(verdicts)):.1%} of scored days)")
print(f"  statistical  auc {comparison.iloc[-1]['stat_auc']}   recall {comparison.iloc[-1]['stat_recall']}")
print(f"  gru          auc {comparison.iloc[-1]['gru_auc']}   recall {comparison.iloc[-1]['gru_recall']}")
for _, line in summary.iterrows():
    print(f"  {line['condition']:<12} prompt   {int(line['sentences_removed'])} sentences removed   "
          f"faithfulness {line['faithfulness_before']} before, 1.0 after")


written to /kaggle/working
  daily_features.csv                9.9 KB
  baseline_median.csv              12.4 KB
  baseline_mad.csv                 11.3 KB
  deviation_scores.csv             31.5 KB
  daily_verdicts.csv                9.6 KB
  detector_comparison.csv           0.2 KB
  verification_report.csv          18.8 KB
  verification_summary.csv          0.2 KB
  zone_map.json                     0.7 KB
  evidence_records.json             4.9 KB
  detector_config.json              1.0 KB
  gru_autoencoder.pt               46.8 KB

summary
  days analysed              218
  alerts raised              15  (7.4% of scored days)
  statistical  auc 0.94   recall 0.551
  gru          auc 0.779   recall 0.355
  production   prompt   0 sentences removed   faithfulness 1.0 before, 1.0 after
  permissive   prompt   83 sentences removed   faithfulness 0.317 before, 1.0 after
